# Data Ingestion Pipeline

This notebook demonstrates a simple ETL pipeline that loads order‑payment data into a MySQL database and product‑category data into a MongoDB collection.

## Overview

| Step | Destination | Source |
|------|-------------|--------|
| 1️⃣ | MySQL table `olist_order_payments` | `../brazilian-ecommerce/olist_order_payments_dataset.csv` |
| 2️⃣ | MongoDB collection `product_categories` | `../brazilian-ecommerce/product_category_name_translation.csv` |

In [3]:
import pandas as pd
import mysql.connector
from mysql.connector import Error

# Connection details
hostname = "45icnr.h.filess.io"
database = "olistproject_hadsnaketo"
port = "3307"
username = "olistproject_hadsnaketo"
password = "4ca25af2ac7add11226e58b0a18e1faad7c3e96d"

# CSV file path
csv_file_path = "../brazilian-ecommerce/olist_order_payments_dataset.csv"

# table name where the data will be uploaded
table_name = "olist_order_payments"

try:
    # establish a connection to MySQL server
    connection = mysql.connector.connect(host=hostname, database=database, user=username, password=password, port=port)
    if connection.is_connected():
        db_Info = connection.get_server_info()
        print("Connected to MySQL Server version ", db_Info)
        # create a cursor to execute SQL queries
        cursor = connection.cursor()
        # drop table if it already exists (for clean insertion)
        cursor.execute(f"DROP TABLE IF EXISTS {table_name}")
        print(f"Table `{table_name}` dropped if it existed.")

        # create a table structure to match CSV file
        create_table_query = f"""
        CREATE TABLE {table_name} (
            order_id VARCHAR(50),
            payment_sequential INT,
            payment_type VARCHAR(20),
            payment_installments INT,
            payment_value FLOAT
        );        
        """

        cursor.execute(create_table_query)
        print(f"Table `{table_name}` created successfully!")

        #load the CSV data into pandas DataFrame
        data = pd.read_csv(csv_file_path)
        print("CSV data loaded info pandas DataFrame.")

        # insert data in batches of 1000 records
        batch_size = 1000
        total_records = len(data)

        print(f"Starting data insertion into `{table_name}` in batches of {batch_size} records.")
        for start in range(0, total_records, batch_size):
            end = start + batch_size
            batch = data.iloc[start:end]

            # convert batch to list of tuples for MySQL insertion
            batch_records = [
                tuple(row) for row in batch.itertuples(index=False, name=None)
            ]

            # prepare the INSERT query
            insert_query = f"""
            INSERT INTO {table_name}
            (order_id, payment_sequential, payment_type, payment_installments, payment_value)
            VALUES (%s, %s, %s, %s, %s)
            """

            # execute the insertion query for the batch
            cursor.executemany(insert_query, batch_records)
            connection.commit()
            print(f"Inserted records {start + 1} to {min(end, total_records)} successfully.")

        print(f"All {total_records} records inserted successfully into `{table_name}`.")

        #record = cursor.fetchone()
        #print("You're connected to database: ", record)

except Error as e:
    print("Error while connecting to MySQL or inserting data", e)
finally:
    if connection.is_connected():
        cursor.close()
        connection.close()
        print("MySQL connection is closed")

/tmp/ipykernel_566459/2038563368.py:22: DeprecationWarning: Call to deprecated function get_server_info. Reason: 
    The property counterpart 'server_info' should be used instead.

  db_Info = connection.get_server_info()


Connected to MySQL Server version  8.0.36-28
Table `olist_order_payments` dropped if it existed.
Table `olist_order_payments` created successfully!
CSV data loaded info pandas DataFrame.
Starting data insertion into `olist_order_payments` in batches of 1000 records.
Inserted records 1 to 1000 successfully.
Inserted records 1001 to 2000 successfully.
Inserted records 2001 to 3000 successfully.
Inserted records 3001 to 4000 successfully.
Inserted records 4001 to 5000 successfully.
Inserted records 5001 to 6000 successfully.
Inserted records 6001 to 7000 successfully.
Inserted records 7001 to 8000 successfully.
Inserted records 8001 to 9000 successfully.
Inserted records 9001 to 10000 successfully.
Inserted records 10001 to 11000 successfully.
Inserted records 11001 to 12000 successfully.
Inserted records 12001 to 13000 successfully.
Inserted records 13001 to 14000 successfully.
Inserted records 14001 to 15000 successfully.
Inserted records 15001 to 16000 successfully.
Inserted records 16

In [4]:
# importing module
from pymongo import MongoClient

hostname = "pl331l.h.filess.io"
database = "olistDataNoSQL_columnwhat"
port = "27018"
username = "olistDataNoSQL_columnwhat"
password = "ba2bb27af44bc36737fa8ee37ecb5e777fca8131"

uri = "mongodb://" + username + ":" + password + "@" + hostname + ":" + port + "/" + database

# Connect with the portnumber and host
client = MongoClient(uri)

# Access database
mydatabase = client[database]


In [5]:
#load the product_category CSV file into a pandas DataFrame
try:
    product_category_df = pd.read_csv("../brazilian-ecommerce/product_category_name_translation.csv")

except FileNotFoundError:
    print("Error: 'product_category_name_translation.csv' not found.")
    exit()      # exit the script if the file is not found


# MongoDB connection details
hostname = "pl331l.h.filess.io"
database = "olistDataNoSQL_columnwhat"
port = "27018"
username = "olistDataNoSQL_columnwhat"
password = "ba2bb27af44bc36737fa8ee37ecb5e777fca8131"

uri = "mongodb://" + username + ":" + password + "@" + hostname + ":" + port + "/" + database

try:
    # establish a connection to MongoDB
    client = MongoClient(uri)
    db = client[database]

    # select the collection (or create if it doesn't exist)
    collection = db["product_categories"]

    # convert the DataFrame to a list of dictionaries for insertion into MongoDB
    data_to_insert = product_category_df.to_dict(orient="records")

    # insert the data into the collection
    collection.insert_many(data_to_insert)

    print("Data uploaded to MongoDB successfully!")

except Exception as e:
    print(f"An error occured: {e}")

finally:
    # close the MongoDB connection
    if client:
        client.close()
    

Data uploaded to MongoDB successfully!
